
# Integración Consumo – Resistencia

Este notebook:
1. Carga los datasets procesados.
2. Alinea grupos antibióticos (mapping AMR ↔ ATC).
3. Integra ambos datasets a nivel país–año–grupo equivalente.
4. Verifica consistencia estructural.
5. Exporta dataset final listo para Tableau.


In [ ]:

import pandas as pd

df_res = pd.read_csv("../data/processed/antimicrobial_resistance_2000_2024_processed.csv")
df_cons = pd.read_csv("../data/processed/antibiotic_consumption_2012_2024_processed.csv")

print("Resistencia:", df_res.shape)
print("Consumo:", df_cons.shape)

print("\nColumnas resistencia:", df_res.columns.tolist())
print("Columnas consumo:", df_cons.columns.tolist())


Resistencia: (6161, 7)
Consumo: (1725, 4)

Columnas resistencia: ['year', 'country_code', 'country', 'bacteria', 'antibiotic_group', 'resistance_percentage', 'atc_group']
Columnas consumo: ['year', 'country', 'antibiotic_group', 'ddd_per_1000_inhabitants_per_day']


In [2]:

# ==========================
# 1️⃣ Mapping grupos antibióticos
# ==========================

mapping_dict = {
    "Fluoroquinolones": "J01M",
    "Aminoglycosides": "J01G",
    "Third-generation cephalosporins": "J01D"
}

df_res["atc_group"] = df_res["antibiotic_group"].map(mapping_dict)

df_res_mapped = df_res.dropna(subset=["atc_group"])

print("Resistencia tras mapping:", df_res_mapped.shape)


Resistencia tras mapping: (4566, 7)


In [3]:

# ==========================
# 2️⃣ Merge por país–año–grupo
# ==========================

df_merge = pd.merge(
    df_res_mapped,
    df_cons,
    left_on=["country", "year", "atc_group"],
    right_on=["country", "year", "antibiotic_group"],
    how="inner"
)

print("Filas tras merge:", df_merge.shape)


Filas tras merge: (3736, 9)


In [4]:

# ==========================
# 3️⃣ Verificación estructural
# ==========================

duplicates = df_merge.duplicated(
    subset=["country", "year", "bacteria", "antibiotic_group_x"]
).sum()

print("Duplicados clave:", duplicates)


Duplicados clave: 0


In [5]:

# ==========================
# 4️⃣ Selección columnas finales
# ==========================

df_final = df_merge[[
    "country",
    "year",
    "bacteria",
    "antibiotic_group_x",
    "resistance_percentage",
    "ddd_per_1000_inhabitants_per_day"
]]

df_final = df_final.rename(columns={
    "country": "Country",
    "year": "Year",
    "bacteria": "Bacteria",
    "antibiotic_group_x": "Antibiotic Group",
    "resistance_percentage": "Resistance (%)",
    "ddd_per_1000_inhabitants_per_day": "Consumption (DDD/1000/day)"
})

df_final["Resistance (%)"] = df_final["Resistance (%)"].round(2)
df_final["Consumption (DDD/1000/day)"] = df_final["Consumption (DDD/1000/day)"].round(2)

df_final.head()


,Country,Year,Bacteria,Antibiotic Group,Resistance (%),Consumption (DDD/1000/day)
0,Bulgaria,2012,Acinetobacter spp.,Aminoglycosides,58.46,0.28
1,Cyprus,2012,Acinetobacter spp.,Aminoglycosides,52.17,0.08
2,Denmark,2012,Acinetobacter spp.,Aminoglycosides,10.39,0.06
3,Greece,2012,Acinetobacter spp.,Aminoglycosides,78.12,0.14
4,France,2012,Acinetobacter spp.,Aminoglycosides,12.95,0.09


In [6]:

# ==========================
# 5️⃣ Exportar dataset integrado
# ==========================

df_final.to_csv(
    "../data/processed/consumption_resistance_integrated_mapped.csv",
    index=False
)

print("Dataset exportado correctamente.")


Dataset exportado correctamente.
